> **AI-generated.** An AI assistant drafted this page, and no maintainer has reviewed it yet. Please report errors on the issue tracker.

# Distributions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TARPS-group/prob-pipe/blob/dev/overhaul/docs/user_guide/03_distributions.ipynb)

This chapter covers single distributions: the parametric families ProbPipe ships, the operations that summarize a distribution or draw from it, and the support and capabilities each distribution declares.
It ends with distributions built from a sampling function or a log density, which represent the models that no family covers.
Read it before building a model from more than one distribution, which [Joint models and conditional distributions](04_joint_models.ipynb) covers.

In [1]:
# On Google Colab, install ProbPipe; elsewhere this cell does nothing.
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    %pip install --quiet "probpipe-core @ git+https://github.com/TARPS-group/prob-pipe.git@dev/overhaul"

In [2]:
import warnings

import jax
import jax.numpy as jnp
from jax.scipy.special import gamma as gamma_function
from jax.scipy.stats import norm

from probpipe import (
    BijectorTransformedDistribution,
    Gamma,
    LogNormal,
    Normal,
    NumericArraySpec,
    Poisson,
    SupportsLogProb,
    SupportsMean,
    SupportsQuantile,
    SupportsSampling,
    SupportsUnnormalizedLogProb,
    SupportsVariance,
    bijector_for,
    distribution,
    log_prob,
    mean,
    positive,
    prob,
    quantile,
    real,
    sample,
    unnormalized_log_prob,
    variance,
    workflow_run,
)

# TensorFlow Probability warns that JAX runs in 32-bit precision, which these examples use.
warnings.filterwarnings("ignore", message=r"Explicitly requested dtype float64")

## Parametric families

ProbPipe ships the standard parametric families, such as `Normal`, `Gamma`, and `Poisson`, and [the API reference](../api/distributions.md) lists them all.
Each family takes as its first argument the component, which names the part of a draw that the family fills, and then the family's parameters.
The label of the distribution is the family's name, unless `label=` sets another.

In [3]:
x = Normal("x", 0.0, 1.0)
g = Gamma("g", 2.0, 3.0)
n = Poisson("n", 4.0)

print("normal distribution:", repr(x))
print("gamma distribution:", repr(g))
print("Poisson distribution:", repr(n))
print("notation of g:", g)

normal distribution: Normal('Normal', component='x', loc=0.0, scale=1.0)
gamma distribution: Gamma('Gamma', component='g', concentration=2.0, rate=3.0)
Poisson distribution: Poisson('Poisson', component='n', rate=4.0)
notation of g: Gamma(g)


The repr of each distribution names its label and its component and then its parameters.
`Gamma("g", 2.0, 3.0)` is labeled `Gamma` and fills the component `g`, and its concentration is 2 and its rate 3.
`print` shows the shorter notation `Gamma(g)`, which is the label followed by the component.

Array parameters give a distribution whose draws are vectors.
The coordinates are independent, and each one has the parameters at its position:

In [4]:
v = Normal("v", jnp.zeros(3), jnp.array([1.0, 2.0, 3.0]))
print("vector normal distribution:", repr(v))
print("shape of one draw:", v.event_shape)
print("declaration of one draw:", v.event_spec)

vector normal distribution: Normal('Normal', component='v', loc=[0.0, 0.0, 0.0], scale=[1.0, 2.0, 3.0])
shape of one draw: (3,)
declaration of one draw: OutputSpec(v=NumericArraySpec(shape=(3,), dtype=float32, support=real))


The **event declaration** `event_spec` describes one draw: the component `v` holds an array of shape `(3,)` of 32-bit floats whose entries can be any real number.
[Terms](01_terms.ipynb) explains these declarations, and [Joint models and conditional distributions](04_joint_models.ipynb) explains how components combine.

## Operations on a distribution

The operations are functions that take the distribution as their first argument.
`sample` draws from a distribution, and `sample_shape` sets how many draws it returns.
Every operation that draws random numbers runs inside `workflow_run(seed=...)`, which makes its draws reproducible.

In [5]:
with workflow_run(seed=0):
    one_draw = sample(x)
    five_draws = sample(n, sample_shape=5)
    draws_of_v = sample(v, sample_shape=(4,))

print("one draw of x:", one_draw.raw())
print("five draws of n:", five_draws.raw())
print("four draws of v, one per row:")
print(draws_of_v.raw())

one draw of x: -0.9115058
five draws of n: [2. 2. 4. 2. 2.]
four draws of v, one per row:
[[-2.9721317  -1.4887556  -3.6221175 ]
 [ 0.79898363  2.409431    3.2525835 ]
 [-2.169697   -5.3292513   1.0098193 ]
 [-0.19793297  1.1412288   1.5215228 ]]


With a `sample_shape`, the draws lead with one axis per entry of the shape, so the four draws of `v` form a 4 × 3 array.
`.raw()` returns the array that a result holds.

`log_prob` evaluates the log density at a value, and `prob` evaluates the density itself.
For a discrete distribution such as `n`, the density is the probability mass:

In [6]:
print("log density of x at 0:", log_prob(x, 0.0).raw())
print("P(n = 3):", prob(n, 3).raw())
print("log density of v at (0, 0, 0):", log_prob(v, jnp.zeros(3)).raw())
print("log density of v at each of its four draws:", log_prob(v, draws_of_v).raw())

log density of x at 0: -0.9189385


P(n = 3): 0.19536662


log density of v at (0, 0, 0): -4.5485754


log density of v at each of its four draws: [ -9.971282   -6.1811714 -10.509134   -4.859577 ]


A draw of `v` is one vector, so its log density is one number, which is the sum of the log densities of the three independent coordinates.
Given the batch of four draws, `log_prob` returns one log density per draw.

`mean`, `variance`, and `quantile` summarize a distribution.
The mean and the variance of a vector distribution are vectors, and `quantile` takes one level or an array of levels:

In [7]:
print("mean of g:", mean(g).raw())
print("variance of g:", variance(g).raw())
print("variance of v:", variance(v).raw())
print("5%, 50%, and 95% quantiles of g:", quantile(g, jnp.array([0.05, 0.5, 0.95])).raw())

mean of g: 0.6666667
variance of g: 0.22222222
variance of v: [1. 4. 9.]


5%, 50%, and 95% quantiles of g: [0.11845391 0.55944955 1.5812877 ]


Each of these results is exact, because the gamma and normal families have formulas for their means, variances, and quantiles.
Where a family has no formula, the operation estimates the result from draws.
The Poisson family has no formula for its quantiles, and `quantile.check` reports how `quantile` would compute them without running it:

In [8]:
for name, law in [("g", g), ("n", n)]:
    report = quantile.check(law, 0.5)
    print(f"median of {name}: route={report.route}, exact={report.exact}")

with workflow_run(seed=1):
    print("median of n, estimated from draws:", quantile(n, 0.5).raw())

median of g: route=closed_form, exact=True
median of n: route=monte_carlo, exact=False


median of n, estimated from draws: 4.0


The route `closed_form` evaluates a formula, and the route `monte_carlo` estimates the result from draws, so its result is approximate.
[How a result was computed](09_how_results_are_computed.ipynb) explains routes and the record that each result keeps of how it was computed.

## Supports

The **support** of a distribution is the set of values a draw can take.
The event declaration records it, and the `support` attribute returns it:

In [9]:
for name, law in [("x", x), ("g", g), ("n", n)]:
    print(f"support of {name}:", law.support)

support of x: real
support of g: positive
support of n: non_negative_integer


A normal draw is any real number, a gamma draw is a positive number, and a Poisson draw is a non-negative integer.
Each support is a constraint object, such as `real` or `positive`, and its `check` method tests whether values lie in it:

In [10]:
values = jnp.array([-1.0, 0.0, 2.5])
print("values:", values)
print("in the support of g:", g.support.check(values))
print("in the support of n:", n.support.check(values))

values: [-1.   0.   2.5]
in the support of g: [False False  True]
in the support of n: [False  True False]


`support` is the support that every array in a draw shares.
A draw of a joint distribution holds several arrays, and `supports` returns the support of each one, keyed by its path, as [Joint models and conditional distributions](04_joint_models.ipynb) shows.

In [11]:
print("support of each array in a draw of v:", v.supports)

support of each array in a draw of v: {'v': real}


## Capabilities

A **capability** is a computation that a distribution provides, such as drawing samples or evaluating its log density.
Each capability has a class, such as `SupportsSampling` or `SupportsLogProb`, and `isinstance` checks whether a distribution claims it:

In [12]:
capabilities = [
    SupportsSampling,
    SupportsLogProb,
    SupportsUnnormalizedLogProb,
    SupportsMean,
    SupportsVariance,
    SupportsQuantile,
]
print(f"{'capability':<30}{'g':>6}{'n':>6}")
for capability in capabilities:
    claims_for_g, claims_for_n = isinstance(g, capability), isinstance(n, capability)
    print(f"{capability.__name__:<30}{claims_for_g!s:>6}{claims_for_n!s:>6}")

capability                         g     n
SupportsSampling                True  True
SupportsLogProb                 True  True
SupportsUnnormalizedLogProb     True  True
SupportsMean                    True  True
SupportsVariance                True  True
SupportsQuantile                True False


Both families draw samples and evaluate their densities, and both have formulas for the mean and the variance.
Only the gamma family claims `SupportsQuantile`, which is why `quantile` estimated the median of `n` from draws.
An operation selects its route from the capabilities that its arguments claim, so a capability that a distribution lacks makes the operation fall back to another route or raise an error.

## A distribution from a sampler or a density

`distribution(sample=..., log_prob=..., unnormalized_log_prob=..., event_spec=..., component=...)` builds a distribution that no family covers.
It takes a sampling function, a log density, or both, together with the declaration of one draw and the name of its component.

A sampling function takes a JAX random key and returns one draw.
Here the draw is the largest of ten standard normal draws:

In [13]:
def largest_of_ten(key: jax.Array) -> jax.Array:
    return jnp.max(jax.random.normal(key, (10,)))


largest = distribution(
    sample=largest_of_ten,
    event_spec=NumericArraySpec((), support=real),
    component="largest",
)
print("distribution:", largest)
print("repr:", repr(largest))
print("declaration of one draw:", largest.event_spec)
for capability in [SupportsSampling, SupportsLogProb, SupportsMean]:
    print(f"claims {capability.__name__}:", isinstance(largest, capability))

distribution: p(largest)
repr: Distribution('p', component='largest', sample=largest_of_ten)
declaration of one draw: OutputSpec(largest=NumericArraySpec(shape=(), dtype=float32, support=real))
claims SupportsSampling: True
claims SupportsLogProb: False
claims SupportsMean: False


The distribution has the default label `p`, so it displays as `p(largest)`.
It claims only `SupportsSampling`.
Construction traced the sampling function in JAX, without drawing, and filled in the dtype of a draw that the declaration left unset.
The summaries take the route `monte_carlo`, which estimates them from 256 draws, and `log_prob` has no route:

In [14]:
with workflow_run(seed=2):
    print("mean of largest, estimated from draws:", mean(largest).raw())
    print("90% interval of largest:", quantile(largest, jnp.array([0.05, 0.95])).raw())
print("route of the mean:", mean.check(largest).route)
print("log_prob applies:", log_prob.check(largest, 1.0).feasible)

mean of largest, estimated from draws: 1.5317918


90% interval of largest: [0.5774564 2.7040527]
route of the mean: monte_carlo
log_prob applies: False


The exact mean of the largest of ten standard normal draws is 1.54, and the estimate is close to it.

A density alone also defines a distribution.
The argument `unnormalized_log_prob` takes a log density up to an additive constant, such as a density whose normalizing constant has no formula.
Here the density is proportional to $\exp(-x^4/4)$, and `label=` names the distribution `quartic`:

In [15]:
def quartic_log_density(x: jax.Array) -> jax.Array:
    return -0.25 * x**4


quartic = distribution(
    unnormalized_log_prob=quartic_log_density,
    event_spec=NumericArraySpec((), support=real),
    component="x",
    label="quartic",
)
print("distribution:", quartic)
print("claims SupportsUnnormalizedLogProb:", isinstance(quartic, SupportsUnnormalizedLogProb))
print("claims SupportsSampling:", isinstance(quartic, SupportsSampling))
print("unnormalized log density at 1:", unnormalized_log_prob(quartic, 1.0).raw())
print("log_prob applies:", log_prob.check(quartic, 1.0).feasible)
print("mean applies:", mean.check(quartic).feasible)

distribution: quartic(x)
claims SupportsUnnormalizedLogProb: True
claims SupportsSampling: False
unnormalized log density at 1: -0.25
log_prob applies: False
mean applies: False


The distribution evaluates its unnormalized log density, and `log_prob`, which needs the normalized density, does not apply.
`mean` does not apply either, because the distribution claims neither a mean nor sampling.

`sample` takes the route `normalize`, which draws from the density with an inference method, and `sample.check` names the method it selects:

In [16]:
print("route and method of sample:", sample.check(quartic).selected.method_name)
with workflow_run(seed=3):
    quartic_draws = sample(quartic, sample_shape=1000).raw()

exact_second_moment = 2 * gamma_function(0.75) / gamma_function(0.25)
print("E[x^2] estimated from 1,000 draws:", jnp.mean(quartic_draws**2))
print("E[x^2] from its formula:", exact_second_moment)

route and method of sample: normalize/blackjax_nuts


E[x^2] estimated from 1,000 draws: 0.66963494
E[x^2] from its formula: 0.67597824


The method `blackjax_nuts` draws with the No-U-Turn Sampler, so the draws are approximate, and their second moment agrees with the formula $2\,\Gamma(3/4)/\Gamma(1/4)$ to within Monte Carlo error.
[Converting and updating distributions](08_conversion_and_updating.ipynb) shows how to keep such draws as a distribution whose mean and quantiles the operations then compute.

Given both a sampler and a normalized log density, the distribution claims both capabilities.
The largest of ten standard normal draws has the density $10\,\varphi(x)\,\Phi(x)^9$, where $\varphi$ and $\Phi$ are the standard normal density and distribution function:

In [17]:
def largest_of_ten_log_density(x: jax.Array) -> jax.Array:
    return jnp.log(10.0) + norm.logpdf(x) + 9 * norm.logcdf(x)


largest_with_density = distribution(
    sample=largest_of_ten,
    log_prob=largest_of_ten_log_density,
    event_spec=NumericArraySpec((), support=real),
    component="largest",
)
for capability in [SupportsSampling, SupportsLogProb]:
    print(f"claims {capability.__name__}:", isinstance(largest_with_density, capability))
print("log density at 1.5:", log_prob(largest_with_density, 1.5).raw())

claims SupportsSampling: True
claims SupportsLogProb: True


log density at 1.5: -0.36364466


## A transformed distribution

`BijectorTransformedDistribution(component, base, bijector)` is the distribution of $f(X)$ for $X$ drawn from `base` and an invertible map $f$ given as `bijector`.
`bijector_for` returns the canonical invertible map from the real numbers onto a support, which for `positive` is the exponential function.
The exponential of a normal variable is log-normal, so the transformed distribution matches `LogNormal`:

In [18]:
exp_of_normal = BijectorTransformedDistribution("y", Normal("z", 0.0, 0.5), bijector_for(positive))
log_normal = LogNormal("y", 0.0, 0.5)

print("support of the transformed distribution:", exp_of_normal.support)
print("log density at 2, transformed distribution:", log_prob(exp_of_normal, 2.0).raw())
print("log density at 2, LogNormal:", log_prob(log_normal, 2.0).raw())
with workflow_run(seed=4):
    print("mean from 256 draws, transformed distribution:", mean(exp_of_normal).raw())
    more_draws = mean.with_options(n_broadcast_samples=10_000)(exp_of_normal)
    print("mean from 10,000 draws, transformed distribution:", more_draws.raw())
print("exact mean, LogNormal:", mean(log_normal).raw())

support of the transformed distribution: positive
log density at 2, transformed distribution: -1.8798445
log density at 2, LogNormal: -1.8798445
mean from 256 draws, transformed distribution: 1.081782


mean from 10,000 draws, transformed distribution: 1.1381361
exact mean, LogNormal: 1.1331484


The two log densities agree, because the transformed distribution evaluates the base log density at the preimage and subtracts the log-determinant of the map's Jacobian there.
The exponential is not affine, so the transformed distribution has no formula for its mean, and `mean` estimates it from 256 draws.
`mean.with_options(n_broadcast_samples=10_000)` estimates it from 10,000 draws, which brings the estimate closer to the exact mean that `LogNormal` computes from its formula.

## See also

- [Distributions and families](../api/distributions.md): the families, `distribution`, `BijectorTransformedDistribution`, and the capability classes.
- [Operations](../api/operations.md): `sample`, `log_prob`, `prob`, `unnormalized_log_prob`, `mean`, `variance`, and `quantile`.
- [Declarations](../api/declarations.md): `NumericArraySpec`, the constraints such as `positive`, and `bijector_for`.
- [Terms](01_terms.ipynb), [Values and records](02_values_and_records.ipynb), [Joint models and conditional distributions](04_joint_models.ipynb), [Converting and updating distributions](08_conversion_and_updating.ipynb), and [How a result was computed](09_how_results_are_computed.ipynb).
- [Tutorial 1](../tutorials/01_first_analysis.ipynb) builds a prior from parametric families, and [tutorial 6](../tutorials/06_simulating.ipynb) fits a model whose likelihood can only be sampled.